# Lab 1: Scikit-Learn Basics and Data Preprocessing
**Student:** Abdiashim Amirov  
**Dataset:** Adult Census Income (Kaggle: `uciml/adult-census-income`)  
**Goal:** Download a dataset from Kaggle and apply the preprocessing methods from `sklearn.preprocessing`.

**Why this dataset?** It has numeric and categorical columns, missing values (marked as `?`), skewed numbers (`capital_gain`) and a binary target (`income`). This lets me try most preprocessing tools on real data.

## 1. Setup

In [ ]:
# pip install kagglehub[pandas-datasets] scikit-learn pandas matplotlib seaborn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

import kagglehub
from kagglehub import KaggleDatasetAdapter

print("scikit-learn version:", sklearn.__version__)
sns.set_theme(style="whitegrid")

## 2. Download the dataset from Kaggle

In [ ]:
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "uciml/adult-census-income",
    "adult.csv",
)

# Column names in this file use dots (for example "education.num"). Replace them with underscores.
df.columns = df.columns.str.replace(".", "_", regex=False)

print("Shape:", df.shape)
df.head()

## 3. First look at the data

In [ ]:
df.info()

In [ ]:
df.describe().T

The file marks missing values with `?` instead of `NaN`. Pandas does not see them as missing, so I convert them first.

In [ ]:
df = df.replace("?", np.nan)

missing = df.isna().sum()
missing[missing > 0].to_frame("missing_values").assign(percent=lambda x: (x.missing_values / len(df) * 100).round(2))

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print("Shape after removing duplicates:", df.shape)

df["income"].value_counts()

## 4. Choose features and target
- **Target:** `income` (`<=50K` or `>50K`).
- **Numeric features:** age, education_num, capital_gain, capital_loss, hours_per_week.
- **Categorical features:** workclass, education, marital_status, occupation, relationship, race, sex, native_country.
- I drop `fnlwgt` (a census sampling weight, not a personal feature).

In [ ]:
target = "income"
numeric_cols = ["age", "education_num", "capital_gain", "capital_loss", "hours_per_week"]
categorical_cols = ["workclass", "education", "marital_status", "occupation",
                    "relationship", "race", "sex", "native_country"]

X = df[numeric_cols + categorical_cols]
y = df[target]
X.shape, y.shape

## 5. Train/test split
I split the data **before** preprocessing. The preprocessing steps learn values (means, categories) only from the training set. This avoids data leakage.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape, " Test:", X_test.shape)

## 6. Preprocessing methods (`sklearn.preprocessing` and friends)
Each method below is applied to a few columns, so I can see what it does.

### 6.1 Missing values: `SimpleImputer`

In [ ]:
from sklearn.impute import SimpleImputer

cat_missing = ["workclass", "occupation", "native_country"]
print("Missing before:", X_train[cat_missing].isna().sum().to_dict())

imputer = SimpleImputer(strategy="most_frequent")
X_train_cat_imputed = pd.DataFrame(
    imputer.fit_transform(X_train[cat_missing]), columns=cat_missing, index=X_train.index
)
print("Missing after: ", X_train_cat_imputed.isna().sum().to_dict())
print("Most frequent values used:", dict(zip(cat_missing, imputer.statistics_)))

### 6.2 Scaling numeric features: `StandardScaler`, `MinMaxScaler`, `RobustScaler`

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

cols = ["age", "hours_per_week"]
scalers = {
    "Original": None,
    "StandardScaler": StandardScaler(),
    "MinMaxScaler": MinMaxScaler(),
    "RobustScaler": RobustScaler(),
}

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, (name, scaler) in zip(axes, scalers.items()):
    values = X_train[cols[0]].values.reshape(-1, 1)
    if scaler is not None:
        values = scaler.fit_transform(values)
    ax.hist(values, bins=30, color="#3E7CB1")
    ax.set_title(f"{name}\n(age)")
plt.tight_layout()
plt.show()

summary = pd.DataFrame({
    name: pd.Series((s.fit_transform(X_train[cols]) if s else X_train[cols].values).ravel()).describe()[["mean", "std", "min", "max"]]
    for name, s in scalers.items()
}).round(2)
summary

- `StandardScaler`: mean 0, standard deviation 1.
- `MinMaxScaler`: all values between 0 and 1.
- `RobustScaler`: uses the median and IQR, so it is less affected by outliers.

### 6.3 Skewed data: `PowerTransformer` and `QuantileTransformer`

In [ ]:
from sklearn.preprocessing import PowerTransformer, QuantileTransformer

col = "hours_per_week"
orig = X_train[[col]]
yj = PowerTransformer(method="yeo-johnson").fit_transform(orig)
qt = QuantileTransformer(output_distribution="normal", n_quantiles=200, random_state=42).fit_transform(orig)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, data, title in zip(axes, [orig, yj, qt], ["Original", "PowerTransformer (Yeo-Johnson)", "QuantileTransformer (normal)"]):
    ax.hist(data, bins=30, color="#1FB5A8")
    ax.set_title(title)
plt.tight_layout()
plt.show()

`capital_gain` is another very skewed column: most people have 0 and a few have very large values. Let me check it.

In [ ]:
print("Share of zeros in capital_gain:", round((X_train["capital_gain"] == 0).mean() * 100, 1), "%")
print("Max capital_gain:", X_train["capital_gain"].max())

### 6.4 `Binarizer` and `KBinsDiscretizer`

In [ ]:
from sklearn.preprocessing import Binarizer, KBinsDiscretizer

# Binarizer: 1 if the person has capital gain, otherwise 0
has_gain = Binarizer(threshold=0).fit_transform(X_train[["capital_gain"]])
print("People with capital gain:", int(has_gain.sum()), "of", len(has_gain))

# KBinsDiscretizer: split age into 5 groups with equal numbers of people
age_bins = KBinsDiscretizer(n_bins=5, encode="ordinal", strategy="quantile")
age_binned = age_bins.fit_transform(X_train[["age"]])
print("Age bin edges:", np.round(age_bins.bin_edges_[0], 1))
pd.Series(age_binned.ravel()).value_counts().sort_index()

### 6.5 Encoding categories: `OneHotEncoder`, `OrdinalEncoder`, `LabelEncoder`

In [ ]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder

# OneHotEncoder: one new 0/1 column for each category (no order)
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
sex_ohe = ohe.fit_transform(X_train[["sex"]])
print("One-hot columns:", ohe.get_feature_names_out())
print(sex_ohe[:5])

In [ ]:
# OrdinalEncoder: categories become numbers with a meaningful order.
# For education, the order is from lowest to highest level.
edu_order = ["Preschool", "1st-4th", "5th-6th", "7th-8th", "9th", "10th", "11th", "12th",
             "HS-grad", "Some-college", "Assoc-voc", "Assoc-acdm", "Bachelors", "Masters",
             "Prof-school", "Doctorate"]
ordinal = OrdinalEncoder(categories=[edu_order])
edu_ord = ordinal.fit_transform(X_train[["education"]])
pd.DataFrame({"education": X_train["education"].head(), "encoded": edu_ord[:5].ravel()})

In [ ]:
# LabelEncoder: only for the target (y)
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)
print("Classes:", le.classes_, "->", list(range(len(le.classes_))))

### 6.6 `PolynomialFeatures` and `Normalizer`

In [ ]:
from sklearn.preprocessing import PolynomialFeatures, Normalizer

poly = PolynomialFeatures(degree=2, include_bias=False)
poly_out = poly.fit_transform(X_train[["age", "hours_per_week"]])
print("New features:", poly.get_feature_names_out())

# Normalizer scales each ROW (not each column) to length 1
norm = Normalizer(norm="l2")
print("Row length after Normalizer:", np.linalg.norm(norm.fit_transform(X_train[["age", "hours_per_week"]])[:3], axis=1))

## 7. Put it all together: `ColumnTransformer` and `Pipeline`
Different columns need different steps, so I combine them in one object.
- Numeric: fill missing values with the median, then `StandardScaler` (and `PowerTransformer` for the skewed `capital_gain` and `capital_loss`).
- Categorical: fill missing values with the most frequent value, then `OneHotEncoder`.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

skewed_cols = ["capital_gain", "capital_loss"]
normal_numeric = ["age", "education_num", "hours_per_week"]

numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
skewed_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("power", PowerTransformer(method="yeo-johnson")),
])
categorical_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, normal_numeric),
    ("skew", skewed_pipe, skewed_cols),
    ("cat", categorical_pipe, categorical_cols),
])

# Fit on TRAIN only, then transform both sets
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

print("Before:", X_train.shape, "-> After:", X_train_prep.shape)
print("Missing values left:", np.isnan(X_train_prep).sum())
feature_names = preprocessor.get_feature_names_out()
print("Number of features:", len(feature_names))
feature_names[:10]

In [ ]:
pd.DataFrame(X_train_prep[:5], columns=feature_names).iloc[:, :8].round(2)

## 8. Quick check: does preprocessing help?
This is a short extra test, not the main goal of the lab. I train the same model on the raw numeric columns and on the fully preprocessed data.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Baseline: only numeric columns, no scaling, missing values filled with 0
base_train = X_train[numeric_cols].fillna(0)
base_test = X_test[numeric_cols].fillna(0)
base_model = LogisticRegression(max_iter=1000).fit(base_train, y_train_enc)
base_acc = accuracy_score(y_test_enc, base_model.predict(base_test))

# Full preprocessing
full_model = LogisticRegression(max_iter=1000).fit(X_train_prep, y_train_enc)
full_acc = accuracy_score(y_test_enc, full_model.predict(X_test_prep))

print(f"Baseline (numeric only, no preprocessing): {base_acc:.3f}")
print(f"With preprocessing (all features):         {full_acc:.3f}")

## 9. Conclusions
1. The raw data had hidden missing values (`?`) that pandas did not detect until I converted them to `NaN`.
2. I split the data first and fitted every preprocessing step on the training set only, to avoid leakage.
3. I used these tools: `SimpleImputer`, `StandardScaler`, `MinMaxScaler`, `RobustScaler`, `PowerTransformer`, `QuantileTransformer`, `Binarizer`, `KBinsDiscretizer`, `OneHotEncoder`, `OrdinalEncoder`, `LabelEncoder`, `PolynomialFeatures`, `Normalizer`, plus `ColumnTransformer` and `Pipeline`.
4. A `ColumnTransformer` inside a `Pipeline` keeps the process clean and repeatable. The same steps can be reused on new data.
5. Preprocessing made the data ready for a model: no missing values, all numbers, and comparable scales.